# Data Preprocessing


## 1. Load the Dataset

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_csv("data/Airline_Delay_Cause.csv")
print("Original shape:", df.shape)

Original shape: (171666, 21)


## 2. Handle Missing Values and Zero-Flight Records

In [2]:
# Drop rows that have any missing value (very small share of the data)
df = df.dropna()

# Drop rows with 0 scheduled arrivals (delay rate would be divide-by-zero)
df = df[df["arr_flights"] > 0]

print("Shape after cleaning:", df.shape)
print("Missing values left:", df.isnull().sum().sum())

Shape after cleaning: (171223, 21)
Missing values left: 0


## 3. Create the Target Column: delay_risk_category

In [3]:
# Delay rate = share of arrivals delayed by 15+ minutes
df["delay_rate"] = df["arr_del15"] / df["arr_flights"]

# Turn the delay rate into a risk category
def get_risk_category(rate):
    if rate < 0.15:
        return "Low Risk"
    elif rate <= 0.25:
        return "Moderate Risk"
    else:
        return "High Risk"

df["delay_risk_category"] = df["delay_rate"].apply(get_risk_category)

# Class counts and percentages
print(df["delay_risk_category"].value_counts())
print(df["delay_risk_category"].value_counts(normalize=True).round(4) * 100)

delay_risk_category
Low Risk         70452
Moderate Risk    62906
High Risk        37865
Name: count, dtype: int64
delay_risk_category
Low Risk         41.15
Moderate Risk    36.74
High Risk        22.11
Name: proportion, dtype: float64


## 4. Feature Engineering: Season from Month

In [4]:
def month_to_season(m):
    if m in (12, 1, 2):
        return "Winter"
    elif m in (3, 4, 5):
        return "Spring"
    elif m in (6, 7, 8):
        return "Summer"
    else:
        return "Fall"

df["season"] = df["month"].apply(month_to_season)
df["season"].value_counts()


season
Summer    44139
Winter    42644
Spring    42439
Fall      42001
Name: count, dtype: int64

### Encode the Season Column


In [5]:
# One new 0/1 column per season
df = pd.get_dummies(df, columns=["season"], dtype=int)

# Check the new season columns
df[["season_Fall", "season_Spring", "season_Summer", "season_Winter"]].head()

,season_Fall,season_Spring,season_Summer,season_Winter
0,0,0,1,0
1,0,0,1,0
2,0,0,1,0
3,0,0,1,0
4,0,0,1,0


## 5. Prevent Data Leakage
These columns describe the **outcome** of the same month, so the model must not see them.  
`arr_del15` and `delay_rate` are also dropped because the target was built from them.  
`carrier_name` and `airport_name` are dropped because they repeat the `carrier` and `airport` codes.

In [6]:
# Delay-cause counts (they add up to arr_del15)
cause_count_cols = ["carrier_ct", "weather_ct", "nas_ct", "security_ct", "late_aircraft_ct"]

# Delay-duration columns (minutes of delay in the same month)
delay_minute_cols = ["arr_delay", "carrier_delay", "weather_delay", "nas_delay",
                     "security_delay", "late_aircraft_delay"]

# Other same-month outcomes
outcome_cols = ["arr_cancelled", "arr_diverted"]

# Columns the target was made from
target_source_cols = ["arr_del15", "delay_rate"]

# Name columns that repeat the code columns
name_cols = ["carrier_name", "airport_name"]

drop_cols = cause_count_cols + delay_minute_cols + outcome_cols + target_source_cols + name_cols
df = df.drop(columns=drop_cols)

print("Remaining columns:", list(df.columns))

Remaining columns: ['year', 'month', 'carrier', 'airport', 'arr_flights', 'delay_risk_category', 'season_Fall', 'season_Spring', 'season_Summer', 'season_Winter']


## 6. Encode the Carrier Column
`carrier` has only 21 values, so one-hot encoding is fine here.

In [7]:
# One new 0/1 column per carrier
df = pd.get_dummies(df, columns=["carrier"], dtype=int)
df.head()

,year,month,airport,arr_flights,delay_risk_category,season_Fall,season_Spring,season_Summer,season_Winter,carrier_9E,...,carrier_NK,carrier_OH,carrier_OO,carrier_QX,carrier_UA,carrier_US,carrier_VX,carrier_WN,carrier_YV,carrier_YX
0,2023,8,ABE,89.0,Low Risk,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
1,2023,8,ABY,62.0,Moderate Risk,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2,2023,8,AEX,62.0,Moderate Risk,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
3,2023,8,AGS,66.0,Moderate Risk,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
4,2023,8,ALB,92.0,Moderate Risk,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0


## 7. Train-Test Split
We split **before** frequency encoding and scaling, so those steps learn only from the training data (no leakage from the test set).

In [8]:
# X = input features, y = target
X = df.drop(columns=["delay_risk_category"])
y = df["delay_risk_category"]

# 80% train, 20% test; stratify keeps the same class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (136978, 29)
Test shape: (34245, 29)


## 8. Frequency Encoding for the Airport Column
`airport` has 395 values, so one-hot encoding would create too many columns. Instead, each airport is replaced by how often it appears in the training data.

In [9]:
# Share of training records for each airport
airport_freq = X_train["airport"].value_counts(normalize=True)

# Replace airport codes with their frequency
X_train["airport_freq"] = X_train["airport"].map(airport_freq)
X_test["airport_freq"] = X_test["airport"].map(airport_freq).fillna(0)  # airports not seen in training get 0

# Drop the original text column
X_train = X_train.drop(columns=["airport"])
X_test = X_test.drop(columns=["airport"])

X_train[["airport_freq"]].head()

,airport_freq
132066,0.002906
48348,0.008162
11699,0.001219
114770,0.006972
74321,0.008162


## 9. Scale the Numerical Columns

### Cap Outliers in arr_flights
The EDA showed that `arr_flights` outliers are real large hub airports, not errors, so they are **capped (winsorized) instead of deleted**.  
Values above the 99th percentile of the training data are set to that value. The IQR upper bound (550) was not used because it would flatten about 12% of the records.  
The cap is learned from the training data only and then applied to the test data.

In [10]:
# Upper cap = 99th percentile of the training data
cap = X_train["arr_flights"].quantile(0.99)
print("arr_flights cap:", cap)

# How many records will be capped
print("Train records capped:", (X_train["arr_flights"] > cap).sum())
print("Test records capped:", (X_test["arr_flights"] > cap).sum())

# Replace values above the cap with the cap (same cap for train and test)
X_train["arr_flights"] = X_train["arr_flights"].clip(upper=cap)
X_test["arr_flights"] = X_test["arr_flights"].clip(upper=cap)

print("Max arr_flights after capping:", X_train["arr_flights"].max())

arr_flights cap: 4927.460000000021
Train records capped: 1370
Test records capped: 327
Max arr_flights after capping: 4927.460000000021


### Scale with StandardScaler

In [11]:
# Numerical columns to scale
num_cols = ["year", "month", "arr_flights", "airport_freq"]

scaler = StandardScaler()

# Fit on training data only, then apply the same scaling to the test data
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train[num_cols].describe().round(2)

,year,month,arr_flights,airport_freq
count,136978.00,136978.00,136978.00,136978.00
mean,-0.00,-0.00,-0.00,0.00
std,1.00,1.00,1.00,1.00
min,-1.92,-1.60,-0.45,-1.97
25%,-0.88,-1.02,-0.38,-0.87
50%,0.16,0.15,-0.31,0.13
75%,0.85,0.73,-0.12,0.88
max,1.54,1.60,6.03,1.47


## 10. Final Check and Save

### Encode the Target Column
The target classes have a natural order (Low < Moderate < High), so each class is mapped to a number in that order.  
A fixed mapping is used instead of `LabelEncoder`, because `LabelEncoder` sorts alphabetically (High = 0, Low = 1, Moderate = 2) and would lose this order.

In [12]:
# Class name -> number (keeps the Low < Moderate < High order)
risk_mapping = {"Low Risk": 0, "Moderate Risk": 1, "High Risk": 2}

# Apply the same mapping to train and test
y_train = y_train.map(risk_mapping)
y_test = y_test.map(risk_mapping)

print("Mapping:", risk_mapping)

Mapping: {'Low Risk': 0, 'Moderate Risk': 1, 'High Risk': 2}


### Final Check

In [13]:
# Final shapes
print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_test:", X_test.shape, " y_test:", y_test.shape)

# Class balance in each set
print("\nTrain classes:\n", y_train.value_counts(normalize=True).round(3))
print("\nTest classes:\n", y_test.value_counts(normalize=True).round(3))

X_train: (136978, 29)  y_train: (136978,)
X_test: (34245, 29)  y_test: (34245,)

Train classes:
 delay_risk_category
0    0.411
1    0.367
2    0.221
Name: proportion, dtype: float64

Test classes:
 delay_risk_category
0    0.411
1    0.367
2    0.221
Name: proportion, dtype: float64


### Save the Processed Data

In [14]:
# Save the processed data for Stage 5 (model training)
train = X_train.assign(delay_risk_category=y_train)
test = X_test.assign(delay_risk_category=y_test)

train.to_csv("data/train_processed.csv", index=False)
test.to_csv("data/test_processed.csv", index=False)
print("Saved train_processed.csv and test_processed.csv")

Saved train_processed.csv and test_processed.csv
